# Candidate-Set Rule On The Saved Production Models

This notebook applies the candidate-set rule to the two saved production models of the next-category recommender, without retraining or changing them.

The target of the recommender is always a category that the household has not recorded yet. The saved models can still rank a category that the household already has, because the training does not mask the documented categories. The rule removes the recorded categories from the ranking at inference time, which is also how the front-end application uses the recommender.

The notebook compares the saved models with and without the rule on the same holdout partition as in `04_fm_inference.ipynb`, together with three reference baselines.

## Worked Example

A household has already recorded Furniture and Electronics. The saved ranking model returns the following probabilities.

| Category | Probability of the model | Rank without the rule | Rank with the rule |
|---|---|---|---|
| Furniture | 0.35 | 1 | recorded, ranked last |
| Electronics | 0.25 | 2 | recorded, ranked last |
| Appliances | 0.15 | 3 | 1 |
| Clothings | 0.10 | 4 | 2 |
| Jewellery and Personal Items | 0.08 | 5 | 3 |
| Other Items | 0.05 | 6 | 4 |
| Vehicles | 0.02 | 7 | 5 |

Without the rule, the Top-3 recommendation is Furniture, Electronics and Appliances, and two of the three are categories that the household already has. With the rule, the Top-3 recommendation is Appliances, Clothings and Jewellery and Personal Items.

Three points explain the rule.

- The target is never a recorded category. A return to a category that the property already recorded creates no new prediction state, so the next category of every state is a new one. A recorded category can therefore never be a correct answer.
- The probability of a recorded category is set to -1 and not to 0. A category with a probability of exactly 0 would otherwise tie with the recorded ones, and the value -1 ranks below every probability.
- The remaining probabilities are not renormalised. They are used for ranking only. If a probability is displayed to a user, the remaining probabilities can be rescaled to sum to one, or only the rank can be shown.

When fewer candidates remain than the Top-k asks for, the Top-k contains all of them. A household with at most three candidates is therefore always correct in Top-3 by construction, and a household with at most two candidates is always correct in Top-2.

## Design

- **Models.** The two saved models of `03_fm_modelling.ipynb`, loaded from `../models` and not changed: the standard Random Forest (500 trees) for the Top-1 recommendation and the regularised Random Forest (800 trees, `min_samples_split=15`) for the ranked Top-2 and Top-3 recommendations. Each model is masked separately.
- **Partition.** The grouped 70% / 15% / 15% partition by `Property_ID` with seed 42, rebuilt in the same way as in `04_fm_inference.ipynb`. Only the holdout test partition is scored. No model is fitted, no setting is tuned and nothing is selected.
- **Recorded categories.** For every row, the set of categories with a documented count above zero in the modelling table (`documented_count_CAT-001` to `documented_count_CAT-007`). The set is checked against `documented_category_count`, and the target is checked not to be in it.
- **Scoring without the rule.** As in notebook 04: the predicted class of the Top-1 model, and the ranked probabilities of the ranking model for Top-2 and Top-3. It must reproduce the final holdout results of notebook 04.
- **Scoring with the rule.** The recorded categories are removed from the ranking of the same probabilities. The predicted class is the highest-ranked remaining category.
- **Baselines.** (1) The class prior of all categories, which ranks the categories by their frequency in the training partition. (2) The same class prior with the recorded categories skipped, which is the main reference for the scores with the rule. (3) The expected score of choosing at random among the candidate categories, `min(k, c) / c` for `c` candidates, which has no F1 score.
- **Descriptive only.** The holdout partition is a single split with a small number of rows, so the numbers show the effect of the rule on the same rows and are not a stable estimate of a gain. The repeated-split evidence is in notebook `03_candidate_set_stability.ipynb` of the hypothesis-testing audits.
- **Nothing is saved.** No file is written and no model or package file is changed. The rule is defined in this notebook and has no trained parameters.

## Setup

The notebook loads the two saved models and the modelling table. The category names are mapped to the category identifiers of the modelling table, because the recorded categories are stored under the identifiers.

In [1]:
# -------------------------------------------------------------------
# Imports And Data Loading
# -------------------------------------------------------------------

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    top_k_accuracy_score,
)
from sklearn.model_selection import GroupShuffleSplit

DATA_PATH = Path("../data/processed/model_state.csv")
MODELS_DIR = Path("../models")

model_state = pd.read_csv(
    DATA_PATH,
    parse_dates=["Prediction_Time"],
)

top1_model = joblib.load(
    MODELS_DIR / "next_category_top1_random_forest.joblib"
)

ranking_model = joblib.load(
    MODELS_DIR / "next_category_ranking_random_forest.joblib"
)

CLASSES = top1_model.named_steps["classifier"].classes_

assert list(CLASSES) == list(
    ranking_model.named_steps["classifier"].classes_
), "The two models must have the same class order."

CATEGORY_ID = dict(
    model_state[
        ["Next_Category", "Next_Category_ID"]
    ].drop_duplicates().values
)

print("DATA AND MODELS LOADED")
print("-" * 60)

print(f"scikit-learn:  {sklearn.__version__}")
print(f"Rows:          {len(model_state):,}")
print(f"Properties:    {model_state['Property_ID'].nunique():,}")
print(f"Categories:    {len(CLASSES)}")

print()
print("Class order of the models and identifiers:")
for name in CLASSES:
    print(f"  {CATEGORY_ID[name]}  {name}")

print()
print(
    "Top-1 model:   ",
    top1_model.named_steps["classifier"].n_estimators,
    "trees",
)
print(
    "Ranking model: ",
    ranking_model.named_steps["classifier"].n_estimators,
    "trees, min_samples_split =",
    ranking_model.named_steps["classifier"].min_samples_split,
)

DATA AND MODELS LOADED
------------------------------------------------------------
scikit-learn:  1.9.1
Rows:          431
Properties:    234
Categories:    7

Class order of the models and identifiers:
  CAT-005  Appliances
  CAT-004  Clothings
  CAT-001  Electronics
  CAT-003  Furniture
  CAT-002  Jewellery and Personal Items
  CAT-006  Other Items
  CAT-007  Vehicles

Top-1 model:    500 trees
Ranking model:  800 trees, min_samples_split = 15


## Recreate The Partition

The original grouped train, validation and test partition is rebuilt in the same way as in `04_fm_inference.ipynb`. The training partition is needed only for the frequencies of the class-prior baselines. The validation partition is not used.

In [2]:
# -------------------------------------------------------------------
# Recreate The Grouped Partition
# -------------------------------------------------------------------

X = model_state.drop(
    columns=[
        "Next_Category_ID",
        "Next_Category",
    ]
).copy()

y = model_state["Next_Category"].copy()
groups = model_state["Property_ID"].copy()

train_splitter = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=42,
)

train_idx, temp_idx = next(
    train_splitter.split(X, y, groups=groups)
)

X_temp = X.iloc[temp_idx].copy()
y_temp = y.iloc[temp_idx].copy()
temp_groups = groups.iloc[temp_idx].copy()

validation_splitter = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=42,
)

val_idx, test_idx = next(
    validation_splitter.split(X_temp, y_temp, groups=temp_groups)
)

X_train = X.iloc[train_idx].copy()
y_train = y.iloc[train_idx].copy()

X_val = X_temp.iloc[val_idx].copy()

X_test = X_temp.iloc[test_idx].copy()
y_test = y_temp.iloc[test_idx].copy()

train_properties = set(X_train["Property_ID"])
val_properties = set(X_val["Property_ID"])
test_properties = set(X_test["Property_ID"])

print("GROUPED PARTITION")
print("-" * 60)

print(f"Train:      {len(X_train):>3} rows | {len(train_properties):>3} properties")
print(f"Validation: {len(X_val):>3} rows | {len(val_properties):>3} properties")
print(f"Test:       {len(X_test):>3} rows | {len(test_properties):>3} properties")

print()
print("Train / Validation overlap:", len(train_properties & val_properties))
print("Train / Test overlap:      ", len(train_properties & test_properties))
print("Validation / Test overlap: ", len(val_properties & test_properties))

# Predictors of the saved models: the same columns as in notebook 03.
X_test_model = X_test.drop(
    columns=["Property_ID", "Customer_ID", "Prediction_Time"],
    errors="ignore",
).copy()

TARGETS = y_test.to_numpy()
N_TEST = len(TARGETS)

GROUPED PARTITION
------------------------------------------------------------
Train:      309 rows | 163 properties
Validation:  67 rows |  35 properties
Test:        55 rows |  36 properties

Train / Validation overlap: 0
Train / Test overlap:       0
Validation / Test overlap:  0


## Rule Functions

The rule has three small functions. `recorded_matrix` reads the recorded categories of every row from the documented counts, `mask_recorded` replaces the probability of every recorded category by -1, and `ranked_candidates` ranks the remaining categories, with ties resolved by the class order (stable ranking). A row with fewer candidates than the Top-k asks for returns all of them. The functions are defined here and are not part of a package.

In [3]:
# -------------------------------------------------------------------
# Rule Functions
# -------------------------------------------------------------------

def recorded_matrix(feature_frame, classes, category_id_by_name):
    """
    Recorded categories of every row as a boolean matrix.

    One column per class, in the column order of the model
    probabilities. True means the category is already recorded
    (documented count above zero).
    """

    columns = [
        f"documented_count_{category_id_by_name[name]}"
        for name in classes
    ]

    return feature_frame[columns].to_numpy(dtype=float) > 0


def mask_recorded(probabilities, recorded):
    """
    Replace the probability of every recorded category by -1.

    The value -1 is below every probability, so a recorded category
    ranks last and never ties with a probability of 0. The remaining
    probabilities are not renormalised: they are used for ranking only.
    """

    probabilities = np.asarray(probabilities, dtype=float)

    if probabilities.shape != recorded.shape:
        raise ValueError(
            "probabilities and recorded must have the same shape."
        )

    return np.where(recorded, -1.0, probabilities)


def ranked_candidates(probabilities, classes, recorded):
    """
    Rank the categories that are not recorded yet, row by row.

    Returns one list of category names per row, best first. Recorded
    categories are not part of the result.
    """

    classes = np.asarray(classes)

    masked = mask_recorded(probabilities, recorded)

    order = np.argsort(-masked, axis=1, kind="stable")

    return [
        [
            str(classes[column])
            for column in row_order
            if not recorded[row, column]
        ]
        for row, row_order in enumerate(order)
    ]

print("RULE FUNCTIONS DEFINED")
print("-" * 60)
print("recorded_matrix, mask_recorded, ranked_candidates")

RULE FUNCTIONS DEFINED
------------------------------------------------------------
recorded_matrix, mask_recorded, ranked_candidates


## Recorded Categories Of The Holdout Rows

The recorded categories of a row are the categories with a documented count above zero. The set is compared with `documented_category_count`, and the target is checked not to be one of the recorded categories. The number of candidate categories per row shows how many rows are decided by construction in Top-2 and Top-3.

In [4]:
# -------------------------------------------------------------------
# Recorded Categories
# -------------------------------------------------------------------

RECORDED = recorded_matrix(X_test, CLASSES, CATEGORY_ID)

CLASS_INDEX = {name: index for index, name in enumerate(CLASSES)}

target_recorded = np.array(
    [RECORDED[row, CLASS_INDEX[TARGETS[row]]] for row in range(N_TEST)]
)

recorded_count = RECORDED.sum(axis=1)
candidate_count = len(CLASSES) - recorded_count

print("RECORDED CATEGORIES OF THE HOLDOUT ROWS")
print("-" * 70)

print(
    "Rows where the number of recorded categories equals "
    "documented_category_count:",
    f"{int((recorded_count == X_test['documented_category_count'].to_numpy()).sum())} of {N_TEST}",
)
print(
    "Rows whose target is already recorded:",
    f"{int(target_recorded.sum())} of {N_TEST}",
)

print()

summary = (
    pd.Series(candidate_count, name="rows")
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis("candidate categories")
    .to_frame()
)
print(summary.to_string())

print()
print(
    f"Rows with at most 3 candidates (Top-3 correct by construction): "
    f"{int((candidate_count <= 3).sum())} of {N_TEST}"
)
print(
    f"Rows with at most 2 candidates (Top-2 correct by construction): "
    f"{int((candidate_count <= 2).sum())} of {N_TEST}"
)

RECORDED CATEGORIES OF THE HOLDOUT ROWS
----------------------------------------------------------------------
Rows where the number of recorded categories equals documented_category_count: 55 of 55
Rows whose target is already recorded: 0 of 55

                      count
candidate categories       
6                        27
5                        11
4                         9
3                         5
2                         2
1                         1

Rows with at most 3 candidates (Top-3 correct by construction): 8 of 55
Rows with at most 2 candidates (Top-2 correct by construction): 3 of 55


## The Rule Applied To The Saved Models

The probabilities of both saved models are computed once. The rule removes the recorded categories from the ranking of the same probabilities. A direct row-by-row filter of the ranking serves as an independent check of the function.

In [5]:
# -------------------------------------------------------------------
# Probabilities, Ranking Without And With The Rule
# -------------------------------------------------------------------

top1_probabilities = top1_model.predict_proba(X_test_model)
ranking_probabilities = ranking_model.predict_proba(X_test_model)

NONE_RECORDED = np.zeros_like(RECORDED)

# Rankings without the rule (all categories) and with the rule.
top1_ranked_all = ranked_candidates(top1_probabilities, CLASSES, NONE_RECORDED)
ranking_ranked_all = ranked_candidates(ranking_probabilities, CLASSES, NONE_RECORDED)

top1_ranked_rule = ranked_candidates(top1_probabilities, CLASSES, RECORDED)
ranking_ranked_rule = ranked_candidates(ranking_probabilities, CLASSES, RECORDED)

# Independent check: filter the full ranking row by row.
def filter_rows(ranked_all, recorded):
    return [
        [name for name in row if not recorded[index, CLASS_INDEX[name]]]
        for index, row in enumerate(ranked_all)
    ]

assert filter_rows(top1_ranked_all, RECORDED) == top1_ranked_rule
assert filter_rows(ranking_ranked_all, RECORDED) == ranking_ranked_rule

# No recorded category in the output, and the length is the number of candidates.
for ranked in (top1_ranked_rule, ranking_ranked_rule):
    assert all(
        len(row) == candidate_count[index]
        for index, row in enumerate(ranked)
    )

# The masked probabilities rank the recorded categories last.
masked_example = mask_recorded(ranking_probabilities, RECORDED)
assert (masked_example[RECORDED] == -1.0).all()

print("RULE CHECK")
print("-" * 60)
print("Function equals the row-by-row filter:  True")
print("No recorded category in the ranking:    True")
print("Recorded categories masked with -1:     True")

RULE CHECK
------------------------------------------------------------
Function equals the row-by-row filter:  True
No recorded category in the ranking:    True
Recorded categories masked with -1:     True


## Reproduction Check Without The Rule

Before the rule is applied, the saved models are scored in the way of notebook 04: the predicted class of the Top-1 model, and the ranked probabilities of the ranking model for Top-2 and Top-3. The numbers must reproduce the final holdout results of notebook 04 (Top-1 accuracy 0.5091, Top-2 accuracy 0.7091 and Top-3 accuracy 0.8364).

In [6]:
# -------------------------------------------------------------------
# Reproduction Check Without The Rule
# -------------------------------------------------------------------

top1_predictions = top1_model.predict(X_test_model)

reproduced_top1 = accuracy_score(TARGETS, top1_predictions)

reproduced_top2 = top_k_accuracy_score(
    TARGETS, ranking_probabilities, k=2, labels=CLASSES,
)
reproduced_top3 = top_k_accuracy_score(
    TARGETS, ranking_probabilities, k=3, labels=CLASSES,
)

REFERENCE = {
    "Top-1": 0.5091,
    "Top-2": 0.7091,
    "Top-3": 0.8364,
}

print("REPRODUCTION OF THE FINAL HOLDOUT RESULTS (NO RULE)")
print("-" * 70)

for label, value in zip(
    REFERENCE, [reproduced_top1, reproduced_top2, reproduced_top3]
):
    print(
        f"{label}:  {value:.4f}  "
        f"({round(value * N_TEST)} of {N_TEST} rows)  "
        f"| notebook 04: {REFERENCE[label]:.4f}"
    )

reproduced = all(
    round(value, 4) == REFERENCE[label]
    for label, value in zip(
        REFERENCE, [reproduced_top1, reproduced_top2, reproduced_top3]
    )
)

print()
print(
    "The unmasked scores reproduce the results of notebook 04:",
    reproduced,
)

# The row-by-row ranking without the rule gives the same hits.
unmasked_top1_hits = np.array(
    [row[0] == target for row, target in zip(top1_ranked_all, TARGETS)]
)
assert unmasked_top1_hits.sum() == round(reproduced_top1 * N_TEST)

REPRODUCTION OF THE FINAL HOLDOUT RESULTS (NO RULE)
----------------------------------------------------------------------
Top-1:  0.5091  (28 of 55 rows)  | notebook 04: 0.5091
Top-2:  0.7091  (39 of 55 rows)  | notebook 04: 0.7091
Top-3:  0.8364  (46 of 55 rows)  | notebook 04: 0.8364

The unmasked scores reproduce the results of notebook 04: True


## Baselines

Three reference values are computed on the same rows. The class-prior baseline ranks the categories by their frequency in the training partition, and the same ranking with the recorded categories skipped is the main reference for the scores with the rule. The third reference is the expected score of choosing at random among the candidate categories, which is `min(k, c) / c` for a row with `c` candidates and has no F1 score.

In [7]:
# -------------------------------------------------------------------
# Baselines
# -------------------------------------------------------------------

class_counts = y_train.value_counts()

prior_order = sorted(
    class_counts.index,
    key=lambda label: (-class_counts[label], label),
)

print("CLASS PRIOR (TRAINING PARTITION)")
print("-" * 60)
for label in prior_order:
    print(f"{class_counts[label]:>4}  {label}")

prior_ranked_all = [list(prior_order) for _ in range(N_TEST)]

prior_ranked_candidates = [
    [
        label
        for label in prior_order
        if not RECORDED[row, CLASS_INDEX[label]]
    ]
    for row in range(N_TEST)
]

random_scores = {
    k: float(np.mean(np.minimum(k, candidate_count) / candidate_count))
    for k in (1, 2, 3)
}

CLASS PRIOR (TRAINING PARTITION)
------------------------------------------------------------
  70  Other Items
  60  Furniture
  57  Appliances
  47  Electronics
  44  Jewellery and Personal Items
  22  Clothings
   9  Vehicles


## Scores Without And With The Rule

Top-1 uses the Top-1 model, Top-2 and Top-3 use the ranking model, as in notebook 04. The F1 scores belong to the predicted class of the Top-1 model. A hit in Top-k means that the target is among the first k categories of the ranking.

In [8]:
# -------------------------------------------------------------------
# Scores Without And With The Rule
# -------------------------------------------------------------------

def hits(ranked, k):
    return np.array(
        [target in row[:k] for row, target in zip(ranked, TARGETS)]
    )

def predicted_class(ranked):
    return [row[0] if row else "none" for row in ranked]

def score_row(top1_ranked, ranking_ranked):
    predicted = predicted_class(top1_ranked)
    return {
        "Top-1": hits(top1_ranked, 1).mean(),
        "Top-2": hits(ranking_ranked, 2).mean(),
        "Top-3": hits(ranking_ranked, 3).mean(),
        "Macro F1": f1_score(
            TARGETS, predicted, average="macro", zero_division=0
        ),
        "Weighted F1": f1_score(
            TARGETS, predicted, average="weighted", zero_division=0
        ),
    }

rows = {
    "model, not masked": score_row(top1_ranked_all, ranking_ranked_all),
    "model, recorded categories masked": score_row(top1_ranked_rule, ranking_ranked_rule),
    "class prior, all categories": score_row(prior_ranked_all, prior_ranked_all),
    "class prior, candidates only": score_row(prior_ranked_candidates, prior_ranked_candidates),
    "random among candidates": {
        "Top-1": random_scores[1],
        "Top-2": random_scores[2],
        "Top-3": random_scores[3],
        "Macro F1": np.nan,
        "Weighted F1": np.nan,
    },
}

result_table = pd.DataFrame(rows).T

print(f"HOLDOUT SCORES ({N_TEST} rows)")
print("-" * 80)
print(result_table.round(4).to_string())

print()
print("Correct rows (model, out of", N_TEST, ")")
print("-" * 80)

counts = pd.DataFrame(
    {
        "model, not masked": [
            int(hits(top1_ranked_all, 1).sum()),
            int(hits(ranking_ranked_all, 2).sum()),
            int(hits(ranking_ranked_all, 3).sum()),
        ],
        "model, recorded categories masked": [
            int(hits(top1_ranked_rule, 1).sum()),
            int(hits(ranking_ranked_rule, 2).sum()),
            int(hits(ranking_ranked_rule, 3).sum()),
        ],
    },
    index=["Top-1", "Top-2", "Top-3"],
).T
print(counts.to_string())

HOLDOUT SCORES (55 rows)
--------------------------------------------------------------------------------
                                    Top-1   Top-2   Top-3  Macro F1  Weighted F1
model, not masked                  0.5091  0.7091  0.8364    0.3223       0.4431
model, recorded categories masked  0.5273  0.7636  0.8727    0.3483       0.4695
class prior, all categories        0.1273  0.3455  0.5455    0.0323       0.0287
class prior, candidates only       0.2364  0.5636  0.8364    0.1637       0.1932
random among candidates            0.2294  0.4406  0.6336       NaN          NaN

Correct rows (model, out of 55 )
--------------------------------------------------------------------------------
                                   Top-1  Top-2  Top-3
model, not masked                     28     39     46
model, recorded categories masked     29     42     48


## What The Rule Changes

For each row, a hit without the rule is compared with a hit with the rule. By construction no row can lose a hit, because the target is never a recorded category. The table also shows how often a saved model ranks a recorded category first.

In [9]:
# -------------------------------------------------------------------
# Change Of The Hits And Of The First Choice
# -------------------------------------------------------------------

def change(ranked_all, ranked_rule, k):
    before = hits(ranked_all, k)
    after = hits(ranked_rule, k)
    return {
        "hits without rule": int(before.sum()),
        "hits with rule": int(after.sum()),
        "gained": int((after & ~before).sum()),
        "lost": int((before & ~after).sum()),
    }

change_table = pd.DataFrame(
    {
        "Top-1 (Top-1 model)": change(top1_ranked_all, top1_ranked_rule, 1),
        "Top-2 (ranking model)": change(ranking_ranked_all, ranking_ranked_rule, 2),
        "Top-3 (ranking model)": change(ranking_ranked_all, ranking_ranked_rule, 3),
    }
).T

assert (change_table["lost"] == 0).all()

first_choice_recorded = {
    "Top-1 model": float(
        np.mean([RECORDED[row, CLASS_INDEX[top1_ranked_all[row][0]]] for row in range(N_TEST)])
    ),
    "ranking model": float(
        np.mean([RECORDED[row, CLASS_INDEX[ranking_ranked_all[row][0]]] for row in range(N_TEST)])
    ),
}

print("CHANGE OF THE HITS (HOLDOUT ROWS)")
print("-" * 80)
print(change_table.to_string())

print()
print("Share of rows whose first choice without the rule is already recorded")
print("-" * 80)
for name, value in first_choice_recorded.items():
    print(f"{name:<15} {value:.1%}  ({round(value * N_TEST)} of {N_TEST} rows)")

# The rows whose Top-3 of the ranking model differs with the rule.
rows_changed = [
    row
    for row in range(N_TEST)
    if ranking_ranked_all[row][:3] != ranking_ranked_rule[row][:3]
]

print()
print(f"Rows whose Top-3 differs with the rule: {len(rows_changed)} of {N_TEST}")

CHANGE OF THE HITS (HOLDOUT ROWS)
--------------------------------------------------------------------------------
                       hits without rule  hits with rule  gained  lost
Top-1 (Top-1 model)                   28              29       1     0
Top-2 (ranking model)                 39              42       3     0
Top-3 (ranking model)                 46              48       2     0

Share of rows whose first choice without the rule is already recorded
--------------------------------------------------------------------------------
Top-1 model     1.8%  (1 of 55 rows)
ranking model   5.5%  (3 of 55 rows)

Rows whose Top-3 differs with the rule: 15 of 55


## Example From The Holdout Rows

One holdout row in which the saved ranking model ranks a recorded category among its first three is shown with the Top-3 without and with the rule. This is the situation of the worked example at the top of the notebook, with the real probabilities of the saved model.

In [10]:
# -------------------------------------------------------------------
# Example Row
# -------------------------------------------------------------------

if rows_changed:
    row = rows_changed[0]
else:
    row = 0

probability_of = dict(zip(CLASSES, ranking_probabilities[row]))

recorded_names = [
    name for name in CLASSES if RECORDED[row, CLASS_INDEX[name]]
]

print("EXAMPLE ROW")
print("-" * 70)

print(f"Property ID:          {X_test['Property_ID'].iloc[row]}")
print(f"Recorded categories:  {', '.join(recorded_names)}")
print(f"Actual next category: {TARGETS[row]}")

print()
print("TOP-3 WITHOUT THE RULE")
for rank, name in enumerate(ranking_ranked_all[row][:3], start=1):
    flag = "  (already recorded)" if RECORDED[row, CLASS_INDEX[name]] else ""
    print(f"{rank}. {name} ({probability_of[name]:.2%}){flag}")

print()
print("TOP-3 WITH THE RULE")
for rank, name in enumerate(ranking_ranked_rule[row][:3], start=1):
    print(f"{rank}. {name} ({probability_of[name]:.2%})")

EXAMPLE ROW
----------------------------------------------------------------------
Property ID:          PRP-000104
Recorded categories:  Furniture, Jewellery and Personal Items, Other Items
Actual next category: Electronics

TOP-3 WITHOUT THE RULE
1. Electronics (32.90%)
2. Other Items (16.27%)  (already recorded)
3. Appliances (14.09%)

TOP-3 WITH THE RULE
1. Electronics (32.90%)
2. Appliances (14.09%)
3. Clothings (9.80%)


## Limits

- The holdout partition is one split with a small number of rows. The numbers compare the saved models with and without the rule on the same rows and are not an estimate of the gain on new households. The repeated-split evidence for the rule is in the candidate-set audit of the hypothesis-testing notebooks, where the gain depends on the model and is largest for the Logistic Regression.
- The rule can only keep or raise a score, because the target is never a recorded category. It does not make the model learn anything new.
- The saved models were trained without the rule. Training on the candidate set directly (learning the next new category) would be a model change and is not tested here.
- With few candidates, Top-2 and Top-3 are correct by construction for some rows, and the class prior with the recorded categories skipped is already high in Top-3. The scores with the rule should be read against that baseline.
- The scores are not comparable with the acceptance rate of the front-end replay in the project documentation.

## Results And Interpretation

All statements are descriptive. The holdout partition is one split of 55 rows, so every number shows the effect of the rule on the same rows and is not a stable estimate of a gain.

**Checks.** The partition reproduces the one of notebook 04 (309 / 67 / 55 rows, 163 / 35 / 36 properties, no overlap). In all 55 holdout rows the recorded categories agree with `documented_category_count`, and the target is never a recorded category (0 of 55). Without the rule the saved models reproduce the final holdout results of notebook 04 exactly: Top-1 28 of 55 (0.5091), Top-2 39 of 55 (0.7091) and Top-3 46 of 55 (0.8364). The function of the rule equals an independent row-by-row filter of the ranking.

**Candidates in the holdout rows.** 27 rows have 6 candidates, 11 have 5, 9 have 4, 5 have 3, 2 have 2 and 1 has 1. Eight rows (14.5%) have at most three candidates, so Top-3 is correct there by construction, and three rows (5.5%) have at most two candidates, so Top-2 is correct there by construction.

**What the rule changes.** With the recorded categories masked, Top-1 rises from 0.5091 to 0.5273 (28 to 29 rows), Top-2 from 0.7091 to 0.7636 (39 to 42 rows) and Top-3 from 0.8364 to 0.8727 (46 to 48 rows). Macro F1 rises from 0.3223 to 0.3483 and Weighted F1 from 0.4431 to 0.4695. No row loses a hit, as the construction requires (gained 1, 3 and 2 rows, lost 0). The size of the change is small because the saved models rarely rank a recorded category first: this happens in 1 of 55 rows for the Top-1 model (1.8%) and in 3 of 55 rows for the ranking model (5.5%). The Top-3 list of the ranking model differs in 15 of 55 rows, which means that a recorded category sits somewhere in the first three places of the ranking in about a quarter of the rows. The example row (PRP-000104, recorded Furniture, Jewellery and Personal Items and Other Items) shows the effect: Other Items is ranked second without the rule and removed with the rule, and Appliances and Clothings move up.

**Against the baselines.** The class prior with the recorded categories skipped reaches 0.2364, 0.5636 and 0.8364 in Top-1, Top-2 and Top-3, and choosing at random among the candidates reaches 0.2294, 0.4406 and 0.6336. The class prior of all categories, which is the baseline of the earlier notebooks, reaches 0.1273, 0.3455 and 0.5455, but it is no longer the right reference once the recorded categories are removed. With the rule, the saved models are above the candidate class prior by about 29 points in Top-1 (0.5273 against 0.2364) and by about 20 points in Top-2 (0.7636 against 0.5636), and by about 4 points in Top-3 (0.8727 against 0.8364). In Top-3 the margin is small: the Top-3 score of the ranking model without the rule (0.8364) is equal to that of the candidate class prior, so the Top-3 result should not be read as evidence of a strong model, and Top-1 and Top-2 are the metrics to report first. The margin in Top-1 is larger here than in the repeated-split audit (about 9 to 14 points for the Random Forest and CatBoost), which is expected in a single split of 55 rows, where the class prior is also lower (0.2364 against 0.3622 on average).

**Reading.** The rule is acceptable as a post-processing step at inference. It never lowers a score, it matches the task definition (the target is always a category that the household has not recorded) and it is the way the front-end application uses the recommender. The gain on the saved production models is small, in line with the order of magnitude of the repeated-split audit for the Random Forest, because these models rarely put a recorded category first. The rule does not make the model learn anything new, and it is not an estimate of the effect in the application.

**Limits.** One split of 55 rows, where one row changes a score by 1.8 points. The numbers do not test whether the rule changes the conclusion about the anonymisation, which is the subject of the repeated-split audit. The rule is applied to the saved models as they are, and training on the candidate set directly was not tried. The scores are not comparable with the acceptance rate of the front-end replay in the project documentation. Any use of the rule in the production inference function (`predict.py`) is a separate change that was not made here.